In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git"
    import ddm4bio  # noqa: F401

# Capstone Preview - Reinforcement Learning: Dosing as a Sequential Decision

The lessons taught you to *represent* data (PCA, autoencoders), *reconstruct* it
(compressed sensing, SHRED), and *predict* from it (regression, SINDy, forecasting).
The **capstone** adds the one data-driven mode the lessons never exercised: learning
to *act*. Many biomedical problems are sequential decisions under feedback -- when to
sample, how to treat, and the canonical case, **how to dose a drug**. Reinforcement
learning (RL) learns a *policy*: a rule from the current state to the next action,
tuned to a long-run objective rather than a one-step fit.

This page is a self-contained, runnable preview of the exact RL machinery the **[capstone
project](capstone.md)** develops on real **warfarin** PK/PD data. We run it here on a
deliberately small, fully checkable problem so you can see the whole loop -- and its
ground-truth validation -- before applying it to a real drug.

## Reading

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed. -- **Chapter 19**
(reinforcement learning: Markov decision processes, value iteration, and Q-learning), the
method previewed here. The capstone as a whole also draws on **Chapter 4** (least-squares
curve fitting) for the PK/PD model fits and **Chapters 15 and 17-18** (SVD/PCA and clustering)
for characterizing inter-patient variability.

*The dataset.* The warfarin PK/PD data -- a single oral dose, 32 subjects as distributed --
traces to O'Reilly & Aggeler (1968), the classic study of warfarin's slow elimination and
prothrombin-complex response, and to Holford (1986), whose pharmacokinetic/pharmacodynamic
model of that data (an elimination half-life of roughly a day and a half, an IC50 near
1.5 mg/L, and prothrombin-complex turnover) is essentially the one the capstone fits. We fetch
it from the `nlmixr2data` R package (GPL >=3).

*The method -- model-informed RL for dosing.* This preview is a teaching-scale version of a
real, active paradigm. Tosca et al. (2024) review it and give a tutorial on coupling
reinforcement learning with population PK/PD models for precision dosing, warfarin among the
cases. Anzabi Zadeh et al. (2022) is the closest single study to this capstone: a warfarin
PK/PD model simulates virtual patients and deep RL learns a dosing policy that outperforms
accepted clinical protocols. Augustin et al. (2023) put regression, deep RL, and PK/PD
modeling head-to-head on warfarin -- an honest reminder that the mechanistic model can win.
The capstone deliberately swaps their deep RL for tabular value iteration and Q-learning, so
the learned policy can be validated against a ground-truth optimum.

Full citations are on the [References](references.md) page.

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()
print(f"ddm4bio version: {ddm4bio.__version__}")

## The setup: a checkable dosing MDP

Dosing suits the course's validate-against-ground-truth discipline exactly. A
discretized one-compartment pharmacokinetic (PK) model is a finite **Markov decision
process (MDP)**, so **value iteration** solves it *exactly* -- the model-based optimal
dosing policy. We can then ask whether a model-*free* learner recovers that policy from
simulated dosing episodes alone, seeing only sampled transitions and never the transition
kernel. To keep it honest the problem is hard: the drug's clearance varies from patient
to patient, so the optimal policy must hedge against overshooting into toxicity.

### The loop: value iteration, then model-free Q-learning

The agent below only ever calls `env.step()` -- it never sees the transition kernel. Value iteration, which *does* use the kernel, is the ground-truth optimum we check it against.

In [ ]:
from ddm4bio.methods.control import PKDosingEnv, policy_value, value_iteration

env = PKDosingEnv()

# Ground truth: value iteration solves the MDP exactly (the model-based optimum).
v_star, pi_star = value_iteration(env.transition_matrix, env.expected_reward, env.gamma)

# Model-free Q-learning, written out. The agent only ever calls env.step() -- it
# never sees the transition kernel. Each step it picks an action (epsilon-greedy),
# samples a transition, and nudges Q toward the temporal-difference target; the
# step size decays with visits so the estimates settle under the random clearance.
# (ddm4bio.methods.control.q_learning packages exactly this loop; the capstone reuses it.)
rng = np.random.default_rng(0)
Q = np.zeros((env.n_states, env.n_actions))
visits = np.zeros((env.n_states, env.n_actions))
episodes, horizon = 3000, 25

for ep in range(episodes):
    eps = max(0.05, 1.0 - ep / (0.7 * episodes))       # explore early, exploit later
    s = int(rng.integers(env.n_states))                # random start concentration
    for _ in range(horizon):
        a = int(rng.integers(env.n_actions)) if rng.random() < eps else int(Q[s].argmax())
        visits[s, a] += 1
        alpha = 0.5 / (1.0 + 0.02 * visits[s, a])      # decaying learning rate
        s_next, reward = env.step(s, a, rng)           # sample a transition (model-free)
        td_target = reward + env.gamma * Q[s_next].max()
        Q[s, a] += alpha * (td_target - Q[s, a])       # temporal-difference update
        s = s_next

pi_q = Q.argmax(axis=1)

# Validation: does the model-free policy reach the model-based optimal value?
v_q = policy_value(pi_q, env.transition_matrix, env.expected_reward, env.gamma)
frac_optimal = float(np.mean(v_q >= v_star - 1e-6))
print(f"states where Q-learning reaches the optimal value: {frac_optimal:.0%}")
print(f"policy agreement with value iteration:              {np.mean(pi_q == pi_star):.0%}")

The model-free learner recovers the exact optimum, and the policy is
*interpretable* -- read it as a dosing rule. The left panel plots dose against
concentration for both the value-iteration optimum and the Q-learned policy (they
coincide); the right panel simulates concentration trajectories under the learned
policy. Notice the hedge: because clearance is uncertain, the optimal rule keeps a
maintenance dose through the *top* of the window rather than risking a drop below
it -- accepting a little toxicity risk to stay therapeutic.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Left: the learned dosing rule vs the exact optimum, over concentration.
axes[0].step(env.states, env.doses[pi_star], where="mid", linewidth=2.6,
             label="value iteration (optimal)")
axes[0].step(env.states, env.doses[pi_q], where="mid", linewidth=1.4,
             linestyle="--", label="Q-learning (learned)")
axes[0].axvspan(env.c_low, env.c_high, color="0.85", label="therapeutic window")
axes[0].set_xlabel("drug concentration")
axes[0].set_ylabel("dose level")
axes[0].set_title("Learned dosing rule = optimal rule")
axes[0].legend(fontsize=8)

# Right: concentration trajectories under the learned policy, from two starts.
rng_traj = np.random.default_rng(0)
axes[1].axhspan(env.c_low, env.c_high, color="0.85")
for start, color, label in [(0, "C0", "from empty"),
                            (env.n_states - 1, "C3", "from toxic")]:
    for _ in range(8):
        traj = env.rollout(pi_q, start, rng_traj, horizon=20)
        axes[1].plot(traj, color=color, alpha=0.35, linewidth=1.0)
    axes[1].plot([], [], color=color, label=label)
axes[1].set_xlabel("dosing step")
axes[1].set_ylabel("drug concentration")
axes[1].set_title("Policy holds the drug in the therapeutic band")
axes[1].legend(fontsize=8)

fig.suptitle("RL dosing on a stochastic one-compartment PK model")
fig;

QC note. The claim is validated the way the course demands: value iteration gives
the *exact* optimal policy, and Q-learning -- which only ever sampled dosing
episodes -- reaches that optimal value on every state. The learned rule loads when
the compartment is empty, tapers as it fills, and hedges at the top of the window
against variable clearance. Two honest limits: the PK model is a toy, discretized,
single-compartment MDP, and this is a *pedagogical* control problem -- real dosing
uses validated nomograms and monitoring, not a learned table.

**Capstone bridge.** The capstone runs this exact value-iteration-then-learn loop
on a real drug: you fit a one-compartment PK model and an anticoagulation PD model
to real **warfarin** data, calibrate the patient-to-patient variability from the
fitted spread, and learn a dosing policy on that data-grounded model -- a genuine
model-driven <-> data-driven hybrid.

## Interpretation

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

block = interpretation_block(
    claim=(
        f"On a stochastic one-compartment PK model, value iteration gives the exact "
        f"optimal dosing policy, and model-free Q-learning -- seeing only sampled "
        f"episodes -- recovers the optimal value on {frac_optimal:.0%} of states and "
        f"the optimal action on {np.mean(pi_q == pi_star):.0%}."
    ),
    limitations_list=[
        "The PK model is a toy, discretized, single-compartment MDP; the capstone fits "
        "a real one to warfarin data and calibrates its variability from the fitted spread.",
        "This is a pedagogical control problem -- real dosing uses validated nomograms "
        "and INR monitoring, not a learned table.",
        "Value iteration needs the transition kernel, so it is the ground-truth check, "
        "not the deployable method; Q-learning is model-free but here learns a small "
        "table -- real state spaces need function approximation (deep RL), beyond this preview.",
    ],
)
show_interpretation(block)